# Table of Contents
1. [Installation](#install)
2. [Configuration](#config)
3. [Compression Baselines](#compression)
4. [Custom Methods](#custom)
5. [Explanation Methods](#explain)
6. [Evaluation](#eval)

<a id='install'></a>
----------
## 1. Install ThinX package
-------------

In [2]:
# for best - create a new environment and then install all dependencies
! pip install -r requirements.txt

# install custom `goodpoints` package
! pip install custom_packages/goodpoints-main

# install custom `sage` package
! pip install custom_packages/sage-main

# install custom `thinx` package
! pip install custom_packages/thinx-main

Obtaining openxai from git+https://github.com/AI4LIFE-GROUP/OpenXAI.git@a18288620464250856b55234266a6d1dabb64656#egg=openxai (from -r requirements.txt (line 112))
  Cloning https://github.com/AI4LIFE-GROUP/OpenXAI.git (to revision a18288620464250856b55234266a6d1dabb64656) to ./src/openxai
  Running command git clone --filter=blob:none --quiet https://github.com/AI4LIFE-GROUP/OpenXAI.git /Users/katebohan/Desktop/bsc-compress-then-explain/src/openxai
  Running command git rev-parse -q --verify 'sha^a18288620464250856b55234266a6d1dabb64656'
  Running command git fetch -q https://github.com/AI4LIFE-GROUP/OpenXAI.git a18288620464250856b55234266a6d1dabb64656
  Resolved https://github.com/AI4LIFE-GROUP/OpenXAI.git to commit a18288620464250856b55234266a6d1dabb64656
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached anyio-4.9

<a id='config'></a>
------
## 2. Load the configuration (dataset and model) for experiments
-----

In [3]:
from thinx.core.data_loader import DataLoader

loader = DataLoader()
# lets's first load a dataset jm1 with dataset id = 1053 and a neural network model trained on the train set
# NOTE: Reproducibility is guaranteed. The train-test split is fixed by the dataset ID.
# Both preprocessing by TabularPreprocessor and model training are deterministic based on the seed inside DataLoader, 
# ensuring identical results for the same (dataset, model) pair.
dataset_name, X_train, y_train, X_test, y_test, nn_model, preprocessor = loader.load_from_openml(
    dataset_id=1053,
    model_name="ann"
)
# lets's now load the same dataset jm1 with dataset id = 1053 and an xgboost model 
dataset_name, X_train, y_train, X_test, y_test, xgb_model, preprocessor = loader.load_from_openml(
    dataset_id=1053,
    model_name="xgboost"
)

# for future faster computations:
X_test = X_test[:30]
y_test = y_test[:30]

Early stopping at epoch 43


So now we have:
1. dataset_name - the original name of the dataset.
2. X_train - samples after preprocessing, on which both XGBoost and NN have been trained.
3. y_train - y_train – target values (labels) corresponding to `X_train`.
4. X_test – preprocessed samples for future explanation process, strictly separate from training but processed identically.
5. y_test - ground truth target values (labels) corresponding to `X_test`.
6. nn_model - the trained Sequential Neural Network (`PyTorchNN` class) with structure: INPUT (60) → Linear (100) → ReLU → Linear (100) → ReLU → Linear (2) → OUTPUT
7. xgb_model - the trained XGBoost model (n_estimators=200). XGBClassifier or XGBRegressor.
8. preprocessor - aan instance of `TabularPreprocessor` – can be used to get some information about preprocessing applied to samples.

You can get some information about the decisions made while preprocessing the data using the `TabularPreprocessor`.

In [4]:
preprocessor.report_

PreprocessReport(dropped_constant=[], dropped_id_like=[], kept_raw_columns=['loc', 'v(g)', 'ev(g)', 'iv(g)', 'n', 'v', 'l', 'd', 'i', 'e', 'b', 't', 'lOCode', 'lOComment', 'lOBlank', 'locCodeAndComment', 'uniq_Op', 'uniq_Opnd', 'total_Op', 'total_Opnd', 'branchCount'], numeric_columns=['loc', 'v(g)', 'ev(g)', 'iv(g)', 'n', 'v', 'l', 'd', 'i', 'e', 'b', 't', 'lOCode', 'lOComment', 'lOBlank', 'locCodeAndComment', 'uniq_Op', 'uniq_Opnd', 'total_Op', 'total_Opnd', 'branchCount'], categorical_columns=[], task_type='classification')

<a id='compression'></a>
------
## 3. Distribution Compression - different baselines
------

### 1. IID Sampling
This method performs Independent and Identically Distributed sampling

* **`target_size`**: Can be set to any positive integer between 1 and `len(X_test)`. 

In [10]:
from thinx.core.preprocessor import Preprocessor

pre = Preprocessor(
    X=X_test.copy(),
    y=y_test.copy(),
    model=None,  # Model is not needed for iid compression
    compression_method="iid",
    seed=0
)

X_comp, y_comp, idx_comp, t_comp = pre.preprocess(target_size=15)

print("--- iid Compression ---")
print("Original size:", X_test.shape[0])
print("Compressed size:", X_comp.shape[0])
print("Selected indices:", idx_comp)
print("Compression time (s):", t_comp)

--- iid Compression ---
Original size: 30
Compressed size: 15
Selected indices: [ 5 10 24 17  6 20 13  0 14  9 29  1 22  4 16]
Compression time (s): 0.0002009868621826172


### 2. ARFPY Sampling
This method leverages Adversarial Random Forests (ARF) to model the underlying data distribution and generate synthetic samples.

* **`target_size`**: Can be set to any positive integer greater than 1. Instead of selecting existing rows, the algorithm generates new artificial samples.

In [11]:
pre_arf = Preprocessor(
    X=X_test.copy(),
    y=y_test.copy(),
    model=None,  # Model is not needed for ARFPY compression
    compression_method="arfpy",
    seed=0
)
X_comp_arf, y_comp_arf, idx_arf, t_arf = pre_arf.preprocess(target_size=4)

print("--- ARFPY Compression ---")
print("Original size:", X_test.shape[0])
print("Compressed size:", X_comp_arf.shape[0])
print("Selected indices:", idx_arf, " Returns -1 for synthetic points")
print("One of selected synthetic points:", X_comp_arf[0])
print("Compression time (s):", t_arf)

--- ARFPY Compression ---
Original size: 30
Compressed size: 4
Selected indices: [-1 -1 -1 -1]  Returns -1 for synthetic points
One of selected synthetic points: [-0.63494578 -0.31320375  0.25653915 -0.19427506 -0.29864743 -0.25509089
 -0.81554777 -0.40685052 -0.18179018 -0.10287723 -0.36771017 -0.08240199
 -0.08831097 -0.43044005 -0.35881241 -0.04159299 -0.1112701  -0.3912405
 -0.3141112  -0.35775559 -0.36622382]
Compression time (s): 0.36480069160461426



### 3. Stein Thinning

This method performs gradient-based compression by iteratively selecting points that minimize the Stein discrepancy to approximate the underlying distribution.

* **`target_size`**: Can be set to any positive integer. The algorithm uses a greedy selection process to pick the most representative samples one by one until the specified size is reached.
* **`grad_type`**: Defaults to `b'gaussian'`, which is the simplest and recommended option. While `b'kde'` and `b'gmm'` are also available, they are **strictly experimental** and we do not recommend them.

In [12]:
pre_stein = Preprocessor(
    X=X_test.copy(),
    y=y_test.copy(),
    model=None,
    compression_method="stein_thinning",
    seed=0
)

X_comp_st, y_com_st, idx_st, t_st = pre_stein.preprocess(
    target_size=15,
    grad_type=b'kde'
)

print("--- Stein Thinning Compression ---")
print("Original size:", X_test.shape[0])
print("Compressed size:", X_comp_st.shape[0])
print("Selected indices:", idx_st)
print("Compression time (s):", t_st)

--- Stein Thinning Compression ---
Original size: 30
Compressed size: 15
Selected indices: [16 22  9 23 14 25 20 15  4 26  2 12 27 19 18]
Compression time (s): 0.0033690929412841797


### 4. Influence-based Compression
This method utilizes Influence Functions (via the `pydvl` library) to identify and select the most "impactful" samples from the dataset based on the model's loss surface. This model requires a pytorch model to be given.

* **`target_size`**: Can be set to any positive integer. The algorithm computes self-influence scores for all points and uses them to sample a representative subset.

In [14]:
pre = Preprocessor(
    X=X_test,
    y=y_test,
    model=nn_model,
    compression_method="influence",
)
X_comp_i, y_comp_i, idx_i, t_comp_i, matrix = pre.preprocess(target_size=15)

print("--- Influence-based Compression ---")
print("Original size:", X_test.shape[0])
print("Compressed size:", X_comp_i.shape[0])
print("Selected indices:", idx_i)
print("Compression time (s):", t_comp_i)

Classification identified
--- Influence-based Compression ---
Original size: 30
Compressed size: 15
Selected indices: [19  8  1  0 24 27 18 22 16 28 25 26  6 21 13]
Compression time (s): 0.19830870628356934


### 5. Kernel Thinning
This method utilizes kernel-based thinning sequences to minimize the Maximum Mean Discrepancy (MMD).

* **`target_size`**: Must be of a power of 2, a posisite integer less than original size, otherwise algorithm will try to find the closest power of 2 based on `target_size`. Important: The possible values are contrainted by `g` - oversampling parameter and `num_bins` - number of bins used in Compress++ algorithms. 
* **`kernel`**: Supports various types including `"gaussian"`, `"sobolev"`, `"inverse_multiquadric"`, and `"matern"`. The kernel choice defines the function space in which the discrepancy between the original and compressed sets is minimized.

In [15]:
# test different sizes and kernels

pre = Preprocessor(
    X=X_test.copy(),
    y=y_test.copy(),
    model=None,
    compression_method="kernel_thinning",
    seed=0
)

X_comp_kt_g, y_com_kt_g, idx_kt_g, t_kt_g = pre.preprocess(
    g=4,
    num_bins=4, 
    target_size=2, 
    kernel="gaussian"
)

print("--- Stein Thinning Compression - Gaussian ---")
print("Original size:", X_test.shape[0])
print("Compressed size:", X_comp_kt_g.shape[0])
print("Selected indices:", idx_kt_g)
print("Compression time (s):", t_kt_g)

X_comp_kt_s, y_com_kt_s, idx_kt_s, t_kt_s = pre.preprocess(
    g=4,
    num_bins=4, 
    target_size=4, 
    kernel="sobolev"
)

print(" \n --- Stein Thinning Compression - Sobolev ---")
print("Original size:", X_test.shape[0])
print("Compressed size:", X_comp_kt_s.shape[0])
print("Selected indices:", idx_kt_s)
print("Compression time (s):", t_kt_s)

X_comp_kt_imq, y_com_kt_imq, idx_kt_imq, t_kt_imq = pre.preprocess(
    g=4,
    num_bins=4, 
    target_size=8, 
    kernel="inverse_multiquadric"
)

print(" \n --- Stein Thinning Compression - Inverse-Multiquadric ---")
print("Original size:", X_test.shape[0])
print("Compressed size:", X_comp_kt_imq.shape[0])
print("Selected indices:", idx_kt_imq)
print("Compression time (s):", t_kt_imq)

X_comp_kt_m, y_com_kt_m, idx_kt_m, t_kt_m = pre.preprocess(
    g=4,
    num_bins=4, 
    target_size=16, 
    kernel="matern"
)

print(" \n --- Stein Thinning Compression - Matern ---")
print("Original size:", X_test.shape[0])
print("Compressed size:", X_comp_kt_m.shape[0])
print("Selected indices:", idx_kt_m)
print("Compression time (s):", t_kt_m)

Gaussian kernel: lambda^2 = 9.155884507140055
--- Stein Thinning Compression - Gaussian ---
Original size: 30
Compressed size: 2
Selected indices: [ 1 11]
Compression time (s): 0.000701904296875
Sobolev kernel: parameters [1. 2. 3.]
 
 --- Stein Thinning Compression - Sobolev ---
Original size: 30
Compressed size: 4
Selected indices: [ 9  9  9 23]
Compression time (s): 0.00030517578125
Inverse-Multiquadric kernel: c = 9.155884507140055
 
 --- Stein Thinning Compression - Inverse-Multiquadric ---
Original size: 30
Compressed size: 8
Selected indices: [ 0  5  1  3 15 13 25 27]
Compression time (s): 0.00026607513427734375
Matérn kernel: parameters [1.         1.51293461 0.5        1.         3.02586922 1.5
 1.         6.05173843 2.5       ]
 
 --- Stein Thinning Compression - Matern ---
Original size: 30
Compressed size: 16
Selected indices: [ 0  1  3  5  7  9 11 13 15 17 19 21 23 25 27 29]
Compression time (s): 0.0001800060272216797


<a id='custom'></a>
-------
## 4. Custom Methods - Stratified and Prediction-Aware
-------
 You can also define for a processor a custom data modification method, which will be applied before compression - none, stratified (only for classification), predictions methods.

* **`data_modification_method="predictions"`**: This method augments the feature matrix $X$ by appending the model's output as additional columns. For **regression** tasks, it appends a single column containing the predicted values. For **classification** tasks, it appends $n$ columns containing the predicted probabilities for each of the $n$ classes. This allows the compression algorithm to account for the model's decision manifold during the thinning process.

In [16]:
pre = Preprocessor(
    X=X_test.copy(),
    y=y_test.copy(),
    model=nn_model, # Model is required (nn_model or xgb_model can be used)
    compression_method="arfpy",
    seed=0,
    data_modification_method="predictions"
)

X_comp_st, y_com_st, idx_st, t_st = pre.preprocess(
    target_size=11,
    grad_type=b'kde'
)

print("--- Arfpy Compression using custom predictons adjustment ---")
print("Original size:", X_test.shape[0])
print("Compressed size:", X_comp_st.shape[0])
print("Selected indices:", idx_st)
print("Compression time (s):", t_st)

--- Arfpy Compression using custom predictons adjustment ---
Original size: 30
Compressed size: 11
Selected indices: [-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
Compression time (s): 0.380018949508667


* **`data_modification_method="stratified"`**: This method performs compression independently across groups defined by the model's predicted classes - compression algorithm is applied to each group separately. 

In [17]:
pre = Preprocessor(
    X=X_test.copy(),
    y=y_test.copy(),
    model=nn_model, # Model is required (nn_model or xgb_model can be used)
    compression_method="stein_thinning",
    seed=0,
    data_modification_method="stratified"
)

X_comp_st, y_com_st, idx_st, t_st = pre.preprocess(target_size=6)

print("--- Stein Thinning Compression using custom stratified adjustment ---")
print("Original size:", X_test.shape[0])
print("Compressed size:", X_comp_st.shape[0])
print("Selected indices:", idx_st)
print("Compression time (s):", t_st)

--- Stein Thinning Compression using custom stratified adjustment ---
Original size: 30
Compressed size: 6
Selected indices: [23 22 25  0  8 28]
Compression time (s): 0.0013561248779296875


<a id='explain'></a>
--------
# 5. Explanation Methods
---------

### 1. SHAP Kernel - local explanations

In [18]:
from thinx.core.explainer import Explainer

In [19]:
explainer = Explainer(
    model=xgb_model, 
    explainer_name="shap",
    strategy="kernel",
    task_type="classification",
    seed=0
)

shap_values, exp_time = explainer.explain(X_foreground=X_test, X_background=X_comp, n_jobs=1)
print("time for explanation (s):", exp_time)
print("shap values for sample 0:", shap_values[0])

Using 1 CPU cores for parallel processing.
Explaining with SHAP. 30 samples to explain using 15 background samples.
Running SHAP explanation in parallel using 1 jobs, 4 batches of size 10
  → Finished batch 1/4
  → Finished batch 2/4
  → Finished batch 3/4
  → Finished all 30 samples
time for explanation (s): 2.3744077682495117
shap values for sample 0: [ 0.07507134 -0.01191757 -0.00711553  0.01051037  0.02290658  0.01188189
  0.00540842 -0.02136389  0.00963464 -0.00240487 -0.00416379  0.00468737
  0.00792533 -0.00272222  0.01158646 -0.00074145  0.00506899  0.0068108
 -0.04534976 -0.02838132 -0.00944881]


### 2. SAGE Permutation - global explanations

In [20]:
explainer = Explainer(
    model=nn_model, 
    explainer_name="sage",
    strategy="permutation",
    task_type="classification",
    seed=0
)

sage_values, exp_time = explainer.explain(X_foreground=X_test, 
                                          X_background=X_comp_arf,
                                          y_background=y_comp_arf, # sage requires background labels
                                          y_foreground=y_test, # sage requires foreground labels
                                          n_jobs=4)
print("time for explanation (s):", exp_time)
print("sage values:", sage_values)

Using 4 CPU cores for parallel processing.
Explaining with SAGE. 30 samples to explain using 4 background samples.
PermutationEstimator will use 4 jobs
StdDev Ratio = 0.0752 (Converge at 0.0250)
StdDev Ratio = 0.0542 (Converge at 0.0250)
StdDev Ratio = 0.0443 (Converge at 0.0250)
StdDev Ratio = 0.0390 (Converge at 0.0250)
StdDev Ratio = 0.0335 (Converge at 0.0250)
StdDev Ratio = 0.0302 (Converge at 0.0250)
StdDev Ratio = 0.0277 (Converge at 0.0250)
StdDev Ratio = 0.0260 (Converge at 0.0250)
StdDev Ratio = 0.0243 (Converge at 0.0250)
Detected convergence
time for explanation (s): 3.7261672019958496
sage values: [ 0.0426855   0.01239541 -0.00979281  0.00190501  0.00301943  0.00270775
 -0.00966227  0.00425865  0.00875306  0.0011895  -0.00130894  0.00101949
  0.00010132 -0.00739588  0.02570433  0.00232545  0.01561863  0.00569683
 -0.00111094  0.00183357  0.00675699]


### 3. SHAP-IQ Kernel - contributions of pairs of features

In [21]:
explainer = Explainer(
    model=nn_model, 
    explainer_name="shapiq",
    strategy="kernel",
    task_type="classification",
    seed=0
)

shapiq_values, exp_time = explainer.explain(X_foreground=X_test, X_background=X_comp_arf, n_jobs=4, verbose=False)
print("time for explanation (s):", exp_time)
print("shapiq values for sample 0:", shap_values[0])

Using 4 CPU cores for parallel processing.
Explaining with ShapIQ. 30 samples to explain using 4 background samples.
time for explanation (s): 5.300970554351807
shapiq values for sample 0: [ 0.07507134 -0.01191757 -0.00711553  0.01051037  0.02290658  0.01188189
  0.00540842 -0.02136389  0.00963464 -0.00240487 -0.00416379  0.00468737
  0.00792533 -0.00272222  0.01158646 -0.00074145  0.00506899  0.0068108
 -0.04534976 -0.02838132 -0.00944881]


### 4. Expected gradients - local explanations - `only for neural networks!`

In [22]:
explainer = Explainer(
    model=nn_model, 
    explainer_name="expected_gradients",
    strategy="na",
    task_type="classification",
    seed=0
)

exp_grad_values, exp_time = explainer.explain(X_foreground=X_test, X_background=X_comp_arf, n_jobs=4, verbose=False)
print("time for explanation (s):", exp_time)
print("expected gradients values for sample 0:", exp_grad_values[0])

Using 4 CPU cores for parallel processing.
Explaining with Expected Gradients. 30 samples to explain using 4 background samples.
time for explanation (s): 0.5335323810577393
expected gradients values for sample 0: tensor([ 1.3379e-01, -1.0715e-02,  2.0536e-02,  2.0328e-02,  2.0233e-02,
         2.2433e-02, -1.7429e-01, -5.0220e-02, -1.2450e-02,  9.8012e-04,
        -3.2224e-03,  1.2944e-04, -4.2850e-02, -4.1779e-03,  6.1804e-02,
         2.4763e-02,  1.2900e-02,  3.1548e-02,  1.8836e-02, -5.9274e-02,
        -7.1972e-02])


<a id='eval'></a>
## 6. Evaluation
-----

### 0. Ground truth explanation calculation

In [23]:
explainer = Explainer(
    model=xgb_model, 
    explainer_name="shap",
    strategy="kernel",
    task_type="classification",
    seed=0
)

shap_values_gt, exp_time_gt = explainer.explain(X_foreground=X_test, X_background=X_test, n_jobs=1)
print("time for explanation (s):", exp_time)
print("shap values for sample 0:", shap_values[0])

Using 1 CPU cores for parallel processing.
Explaining with SHAP. 30 samples to explain using 30 background samples.
Running SHAP explanation in parallel using 1 jobs, 4 batches of size 10
  → Finished batch 1/4
  → Finished batch 2/4
  → Finished batch 3/4
  → Finished all 30 samples
time for explanation (s): 0.5335323810577393
shap values for sample 0: [ 0.07507134 -0.01191757 -0.00711553  0.01051037  0.02290658  0.01188189
  0.00540842 -0.02136389  0.00963464 -0.00240487 -0.00416379  0.00468737
  0.00792533 -0.00272222  0.01158646 -0.00074145  0.00506899  0.0068108
 -0.04534976 -0.02838132 -0.00944881]


In [24]:
from thinx.core.evaluation import Evaluator

evaluator = Evaluator(ground_truth_explanation=shap_values_gt, ground_truth_points=X_test)

print("--- Evaluation of ARFPY compressed applied to SHAP explanations ---")
metrics_comp_eval = evaluator.evaluate_compression(
    compressed_points=X_comp_arf,
)
metrics_exp_eval = evaluator.evaluate_explanation(
    explanation=shap_values, 
    time_elapsed=t_arf,
    num_samples=len(X_comp_arf)
)
print("Compression evaluation metrics:", metrics_comp_eval) 
print("Explanation evaluation metrics:", metrics_exp_eval)


--- Evaluation of ARFPY compressed applied to SHAP explanations ---
Compression evaluation metrics: {'mmd': 0.10370984200870126}
Explanation evaluation metrics: {'mae': 0.003873645320989183, 'top_k': 0.8600000000000001, 'explanation_time': 0.36480069160461426, 'size': 4}
